# Melanoma Classification Using the Composite DermoscopyDataset

This notebook walks through the full PyHealth pipeline for binary melanoma classification using dermoscopic images.
It combines three publicly available datasets — **ISIC 2018**, **HAM10000**, and **PH2** — into a single composite
dataset via the `DermoscopyDataset` class.

A key feature of this integration is **mode-based image processing** using the `DermoscopyImageProcessor`.
Instead of always passing the raw image to the model, you can choose to feed:
- `whole` — the full dermoscopic image
- `lesion` — only the lesion region (background zeroed out using the segmentation mask)
- `background` — only the surrounding skin (lesion zeroed out)

This is useful for understanding whether a model is relying on the actual lesion or on background artifacts
(e.g., rulers, gel bubbles, dark corners), which is a key question in the paper this repo accompanies.

---

## Step 0: Install PyHealth

In [ ]:
%pip install ipywidgets pyhealth openpyxl

## Step 0a: Colab Setup

This cell clones the `derm_set` branch of the PyHealth repo and installs it in editable mode,
so any changes you make to the source take effect immediately without reinstalling.

**Persistent data (recommended):** Mounting Google Drive lets you keep the downloaded datasets
across Colab sessions so you do not have to re-download them each time.

> Skip the Drive mount block if you are running locally — the data paths in Step 1 will point
> to your local folder instead.

In [ ]:
import os

# ── Option A: running in Google Colab ─────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython().extension_manager.loaded)

if IN_COLAB:
    # 1. Mount Google Drive for persistent dataset storage
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Clone the derm_set branch (skip if already cloned)
    if not os.path.exists('/content/PyHealth'):
        !git clone -b derm_set https://github.com/deadlywrong/PyHealth.git /content/PyHealth

    # 3. Install in editable mode so local edits take effect immediately
    !pip install -e /content/PyHealth --quiet

    PYHEALTH_ROOT = '/content/PyHealth'
    # Dataset root inside Google Drive — change the subfolder name as needed
    filename = "/content/drive/MyDrive/dermoscopy_data.zip"

    with zipfile.ZipFile(filename, 'r') as zipp:
      zipp.extractall('/content/') # Extract to local Colab disk
    DRIVE_DATA_ROOT = '/content/dermoscopy_data'
    os.makedirs(DRIVE_DATA_ROOT, exist_ok=True)

# ── Option B: running locally ──────────────────────────────────────────────────
else:
    # Install from the local clone (editable)
    import subprocess, sys
    repo_dir = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', repo_dir, '--quiet'], check=True)
    PYHEALTH_ROOT = repo_dir
    DRIVE_DATA_ROOT = None   # will use LOCAL_DATA_ROOT in Step 1

print('Setup complete.')
print(f'PyHealth root : {PYHEALTH_ROOT}')
if DRIVE_DATA_ROOT:
    print(f'Drive data root: {DRIVE_DATA_ROOT}')

## Step 1: Configure Data Paths

**Please rename and restructure data in folder `dermoscopy_data/` like this:**
```
dermoscopy_data/
├── dermoscopy-metadata-pyhealth.csv  # ← generated by prepare_metadata()
├── ham10000/
│   ├── images/                     # ISIC_*.jpg  (10 015 files)
│   ├── masks/                      # ISIC_*_segmentation.png  (10 015 files)
│   └── images/metadata.csv         # raw — columns: isic_id, diagnosis_1, ...
├── isic2018/
│   ├── images/                     # ISIC_*.jpg  (2 594 files)
│   ├── masks/                      # ISIC_*_segmentation.png  (2 594 files)
│   └── images/metadata.csv         # raw — columns: isic_id, diagnosis_1, ...
└── ph2/
    ├── PH2 Dataset images/         # original nested BMP structure
    │   ├── IMD002/
    │   │   ├── IMD002_Dermoscopic_Image/IMD002.bmp
    │   │   └── IMD002_lesion/IMD002_lesion.bmp
    │   └── ...
    └── PH2_dataset.txt             # raw pipe-delimited metadata

```

**Label encoding:**

| Dataset | Raw label column | Values | Binary mapping |
|---------|-----------------|--------|----------------|
| HAM10000 | `diagnosis_1` | `'Malignant'` / `'Benign'` / `'Indeterminate'` | Malignant→1, Benign→0, Indeterminate dropped |
| ISIC 2018 | `diagnosis_1` | `'Malignant'` / `'Benign'` / `'Indeterminate'` | Malignant→1, Benign→0, Indeterminate dropped |
| PH2 | `Clinical Diagnosis` in `.txt` | `0` (benign) / `1` (atypical) / `2` (melanoma) | 2→1, 0 or 1→0 |

In [ ]:
import torch
import os
import pandas as pd

# ── Set your dataset root here ────────────────────────────────────────────────
# Colab users: set this to DRIVE_DATA_ROOT (defined in Step 0a)
# Local users: set this to the absolute path of your dermoscopy_data folder
#   e.g. "/Users/yourname/Documents/.../dermoscopy_data"
DATA_ROOT = DRIVE_DATA_ROOT if (DRIVE_DATA_ROOT and os.path.isdir(DRIVE_DATA_ROOT)) \
            else "/Users/ziquanwang/Documents/UIUC MCS/CS 598 DLH/Project/dermoscopy_data"
# ─────────────────────────────────────────────────────────────────────────────

print(f"DATA_ROOT = {DATA_ROOT}")
assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT}"

# Processing mode: one of "whole", "lesion", "background"
MODE = "whole"

# Training hyperparameters
BATCH_SIZE    = 32
NUM_EPOCHS    = 1
LEARNING_RATE = 1e-4

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available(): # for Apple Silicon GPUs
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"\nDevice          : {DEVICE}")
print(f"Processing mode : {MODE}")

## Step 2: Load the Composite Dataset

In [ ]:
import shutil, os

# ── Optional: clear the dataset cache ────────────────────────────────────────
# Run this cell whenever you change the `datasets=` list or suspect a stale
# cache is being used. It deletes everything inside DATA_ROOT/cache/ so the
# next run rebuilds from the freshly written metadata CSV.

cache_dir = os.path.join(DATA_ROOT, "cache")

if os.path.isdir(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cache cleared: {cache_dir}")
else:
    print(f"No cache to clear (directory does not exist): {cache_dir}")

In [ ]:
from pyhealth.datasets import DermoscopyDataset

cache_dir=os.path.join(DATA_ROOT, "cache")

base_dataset = DermoscopyDataset(
    root=DATA_ROOT,
    cache_dir=cache_dir,
    # To use a subset of datasets, pass e.g. datasets=["isic2018", "ham10000"]
)

base_dataset.stats()

### Inspect the unified metadata

The dataset combines all three sources into a single CSV with a `source_dataset` column.

In [ ]:
import pandas as pd
import os

meta = pd.read_csv(os.path.join(DATA_ROOT, "dermoscopy-metadata-pyhealth.csv"))
print("Metadata shape:", meta.shape)
print("\nSource distribution:")
print(meta.source_dataset.value_counts())
print("\nLabel distribution:")
print(meta.label.value_counts().rename({0: "benign", 1: "melanoma"}))
meta.head()

## Step 3: Visualize the Three Processing Modes

The `DermoscopyImageProcessor` can apply three modes using segmentation masks:
- **whole**: full image (no masking)
- **lesion**: only the lesion (background zeroed out)
- **background**: only the skin/background (lesion zeroed out)

This lets you test whether a classifier is learning from the lesion itself or from surrounding skin artifacts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyhealth.processors import DermoscopyImageProcessor

# Pick a sample row from metadata to visualize
sample_row = meta.iloc[0]
img_path  = sample_row["image_path"]
mask_path = sample_row["mask_path"]
label_str = "melanoma" if sample_row["label"] == 1 else "benign"
source    = sample_row["source_dataset"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"Source: {source} | Label: {label_str}", fontsize=13)

for ax, mode in zip(axes, ["whole", "lesion", "background"]):
    proc = DermoscopyImageProcessor(mode=mode, normalize=False)
    tensor = proc.process((img_path, mask_path))  # (3, H, W) in [0, 1]
    img_np = tensor.permute(1, 2, 0).numpy()       # (H, W, 3)
    ax.imshow(img_np)
    ax.set_title(f'mode="{mode}"', fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 4: Set Task and Build DataLoaders

The `DermoscopyMelanomaClassification` task maps each image+mask pair to a binary label.
Passing a custom `DermoscopyImageProcessor` allows you to switch modes.

In [ ]:
from pyhealth.tasks import DermoscopyMelanomaClassification
from pyhealth.processors import DermoscopyImageProcessor
from pyhealth.datasets import get_dataloader, split_by_sample

ph2_task    = DermoscopyMelanomaClassification("ph2")

# Choose the processing mode — swap this to "lesion" or "background" to compare
image_processor = DermoscopyImageProcessor(
    mode=MODE,
    image_size=224,
    normalize=True,   # ImageNet normalization
)

ph2_samples = base_dataset.set_task(
    task=ph2_task,
    input_processors={"image": image_processor},
)


print(f"Total samples: {len(ph2_samples)}")
print(f"Input schema:  {ph2_samples.input_schema}")
print(f"Output schema: {ph2_samples.output_schema}")

In [ ]:
train_dataset, val_dataset, test_dataset = split_by_sample(ph2_samples, [0.7, 0.1, 0.2])

train_loader = get_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = get_dataloader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = get_dataloader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## Step 5: Define the Model

We use `TorchvisionModel` with ResNet50 pretrained on ImageNet — the same architecture used in the
`dermatology_melanoma_classification.py` script from the dermoscopic_artifacts repo.

You can swap `model_name` to any supported architecture:
- `"resnet50"` (default here)
- `"densenet121"` 
- `"vit_b_16"` (Vision Transformer — recommended for ablation study)

### Code Example

```python
# DenseNet121
model = TorchvisionModel(dataset=samples, model_name="densenet121", model_config={"weights": "DEFAULT"})

# Vision Transformer (ViT)
model = TorchvisionModel(dataset=samples, model_name="vit_b_16", model_config={"weights": "DEFAULT"})
```


In [ ]:
from pyhealth.models import TorchvisionModel

model = TorchvisionModel(
    dataset=ph2_samples,
    model_name="resnet50",
    model_config={"weights": "DEFAULT"},   # ImageNet pretrained weights
)

## Step 6: Train the Model

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    device=DEVICE,
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=NUM_EPOCHS,
    optimizer_params={"lr": LEARNING_RATE},
    monitor="roc_auc",
)

## Step 7: Evaluate the Model

In [ ]:
results = trainer.evaluate(test_loader)
print(f"\n=== Test results (mode='{MODE}') ===")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")

## Step 8: Compare Modes (Artifact Bias Analysis)

The core research question is: **is the model relying on the lesion or on background artifacts?**

The cell below trains separate models for each mode and compares their AUC. If `background` AUC is
unexpectedly high, it suggests the model has learned spurious correlations from skin artifacts.

> ⚠️ This cell trains 3 full models and may take a while. Reduce `MODE_NUM_EPOCHS` or run one mode at a time.

In [ ]:
from pyhealth.models import TorchvisionModel
from pyhealth.trainer import Trainer
from pyhealth.datasets import get_dataloader
from pyhealth.processors import DermoscopyImageProcessor
from sklearn.model_selection import KFold

MODE_NUM_EPOCHS = 10
MODE = ["whole", "lesion", "background"]  # change image modes
MODEL_NAME = "resnet50" # can change to "resnet18" or others supported by TorchvisionModel for ablation study

mode_results = {}

isic2018_task = DermoscopyMelanomaClassification("isic2018")
ham10000_task = DermoscopyMelanomaClassification("ham10000")

ham10000_samples = base_dataset.set_task(
    task=ham10000_task,
    input_processors={"image": image_processor},
)

isic2018_samples = base_dataset.set_task(
    task=isic2018_task,
    input_processors={"image": image_processor},
)

for mode in MODE:
    print(f"\n{'='*100}")
    print(f" Training with mode='{mode}'")
    print(f"{'='*100}")

    # Re-process with this mode's processor
    mode_samples = base_dataset.set_task(
        task=ph2_task,
        input_processors={"image": DermoscopyImageProcessor(mode=mode)},
    )
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(mode_samples)))):
        train_dataset = mode_samples.subset(train_idx)
        val_dataset = mode_samples.subset(val_idx)

        tr_l = get_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        va_l = get_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        te_l1 = get_dataloader(isic2018_samples, batch_size=BATCH_SIZE, shuffle=False)
        te_l2 = get_dataloader(ham10000_samples, batch_size=BATCH_SIZE, shuffle=False)

        m = TorchvisionModel(
            dataset=mode_samples,
            model_name=MODEL_NAME,
            model_config={"weights": "DEFAULT"},
        )

        t = Trainer(model=m, device=DEVICE, enable_logging=False)
        t.train(tr_l, va_l, epochs=MODE_NUM_EPOCHS,
                optimizer_params={"lr": LEARNING_RATE}, monitor="roc_auc")
        mode_results[mode + f"_{fold}" + "_ph2"] = t.evaluate(va_l)
        mode_results[mode + f"_{fold}" + "_isic2018"] = t.evaluate(te_l1)
        mode_results[mode + f"_{fold}" + "_ham10000"] = t.evaluate(te_l2)

print("\n" + "="*50)
print(" Mode comparison summary")
print("="*50)
print(f"{'Mode':<15} {'ROC-AUC':>10} {'PR-AUC':>10} {'F1':>8}")
print("-"*45)
for mode, res in mode_results.items():
    roc = res.get("roc_auc", float("nan"))
    pr  = res.get("pr_auc",  float("nan"))
    f1  = res.get("f1",       float("nan"))
    print(f"{mode:<15} {roc:>10.4f} {pr:>10.4f} {f1:>8.4f}")

Output the ROC_AUC results like 'Table 2' in A Study of Artifacts on Melanoma Classification under Diffusion-Based Perturbations for comparison.

In [ ]:
import numpy as np
from collections import defaultdict

# Group roc_auc values by (mode, dataset) across folds
# Keys are formatted as "{mode}_{fold}_{dataset}", e.g. "whole_0_ph2"
grouped = defaultdict(list)
for key, res in mode_results.items():
    parts = key.rsplit("_", 1)          # split off the dataset suffix
    dataset = parts[-1]                  # e.g. "ph2" or "isic2018"
    mode    = parts[0].rsplit("_", 1)[0] # strip the fold index, e.g. "whole"
    grouped[(mode, dataset)].append(res.get("roc_auc", float("nan")))

print(f"{'Mode':<12} {'Dataset':<12} {'Mean ROC-AUC':>14} {'Std ROC-AUC':>13}")
print("-" * 55)
for (mode, dataset), values in sorted(grouped.items()):
    arr = np.array(values)
    print(f"{mode:<12} {dataset:<12} {arr.mean():>14.4f} {arr.std():>13.4f}")
